# **Import Library**

In [1]:
!pip install wandb -q

import os
import copy
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import wandb
os.environ["WANDB_API_KEY"] = "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"

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd

from torchvision import transforms, datasets

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

wandb.login()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: devianestnarendra (devianestnarendra-universitas-darussalam-gontor) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# **Dataset Path**

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

TRAIN_DIR = "/kaggle/input/datasets/shubhamgoel27/dermnet/train"
TEST_DIR  = "/kaggle/input/datasets/shubhamgoel27/dermnet/test"

cuda


# **Train Augmentation**

In [3]:
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

# **Validation Transform**

In [4]:
eval_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# **Load Filepaths**

In [5]:
classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

num_classes = len(classes)

filepaths = []
labels    = []

for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", classes)

Total Images : 15557
Classes      : ['Acne and Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Atopic Dermatitis Photos', 'Bullous Disease Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Herpes HPV and other STDs Photos', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Nail Fungus and other Nail Disease', 'Poison Ivy Photos and other Contact Dermatitis', 'Psoriasis pictures Lichen Planus and related diseases', 'Scabies Lyme Disease and other Infestations and Bites', 'Seborrheic Keratoses and other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors', 'Vasculitis Photos', 'Warts Molluscum and other Viral Infections']


# **Dataset Class**

In [6]:
class SkinDataset(Dataset):

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# **Model Architecture — DermNetCNN**


In [7]:
class ConvBlock(nn.Module):
    """Conv -> BatchNorm -> PReLU -> MaxPool block."""

    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.conv  = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn    = nn.BatchNorm2d(out_channels)
        self.act   = nn.PReLU()
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.pool(x)
        return x


class DermNetCNN(nn.Module):
    """
    Custom CNN sesuai TABEL I — Hyperparameter Configuration of DermNet-CNN:
      - Image size            : 224 x 224
      - Convolution + Maxpool : 4 layer
      - Activation            : PReLU
      - Flatten layer         : Global Max Pooling
      - FC head menyesuaikan num_classes
    """

    def __init__(self, num_classes, in_channels=3):
        super().__init__()

        # 4 blok Conv + MaxPool (sesuai tabel: "Convolution and Maxpool layer: 4, 4")
        self.block1 = ConvBlock(in_channels, 32)    # 224 -> 112
        self.block2 = ConvBlock(32, 64)             # 112 -> 56
        self.block3 = ConvBlock(64, 128)            # 56  -> 28
        self.block4 = ConvBlock(128, 256)           # 28  -> 14

        # Global Max Pooling -> flatten ke (batch, 256)
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.PReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.global_max_pool(x)   # (batch, 256, 1, 1)
        x = torch.flatten(x, 1)       # (batch, 256)

        x = self.fc(x)
        return x


# **Early Stopping & K-Fold**

In [8]:
class EarlyStopping:

    def __init__(self, patience=5):
        self.patience  = patience
        self.best_loss = np.inf
        self.counter   = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# **Wandb**

In [9]:
BATCH_SIZE      = 32
EPOCHS          = 100
EXPERIMENT_NAME = "EXP05_DermNet"

run = wandb.init(
    project = "SkinDisease-ResNet50",
    name    = EXPERIMENT_NAME,
    config  = {
        "architecture"   : "DermNetCNN (custom, 4 Conv+MaxPool blocks)",
        "image_size"     : 224,
        "n_folds"        : 5,
        "epochs"         : EPOCHS,
        "batch_size"     : BATCH_SIZE,
        "optimizer"      : "NAdam",
        "lr"             : 7e-4,
        "weight_decay"   : 1e-4,
        "label_smoothing": 0.1,
        "activation"     : "PReLU",
        "pooling"        : "MaxPool (conv blocks) + GlobalMaxPool (flatten)",
        "loss"           : "CrossEntropyLoss (Categorical Crossentropy)"
    }
)

print(f"WandB Run : {run.name}")
print(f"URL       : {run.url}")
wandb.run.log_code(".")


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260624_091832-35m5mf6w
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run EXP05_DermNet
wandb: ⭐️ View project at https://wandb.ai/devianestnarendra-universitas-darussalam-gontor/SkinDisease-ResNet50
wandb: 🚀 View run at https://wandb.ai/devianestnarendra-universitas-darussalam-gontor/SkinDisease-ResNet50/runs/35m5mf6w
wandb: WARNING No relevant files were detected in the specified directory. No code will be logged to your run.


WandB Run : EXP05_DermNet
URL       : https://wandb.ai/devianestnarendra-universitas-darussalam-gontor/SkinDisease-ResNet50/runs/35m5mf6w


# **Training Loop**

In [10]:

fold_results = []
fold_accuracies    = []
fold_precision     = []
fold_recall        = []
fold_f1            = []
all_fold_best_paths = []

all_train_losses = {}
all_val_losses   = {}


for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):

    print(f"\n{'='*50}")
    print(f"  FOLD {fold + 1} / 5")
    print(f"{'='*50}")

    best_val_loss   = np.inf
    best_train_loss = np.inf
    best_model_path = None

    # ── SPLIT ─────────────────────────────────────────────────────────────────
    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i]    for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i]    for i in val_idx]

    # ── DATALOADER ────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, transform=train_tf),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=0
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, transform=eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    # ── MODEL ─────────────────────────────────────────────────────────────────
    # DermNetCNN custom (4 Conv+MaxPool block, PReLU, Global Max Pooling)
    # sudah termasuk FC head di dalam definisi class -> tidak perlu di-override lagi
    model = DermNetCNN(num_classes).to(device)

    # ── LOSS / OPTIMIZER / SCHEDULER ─────────────────────────────────────────
    class_counts  = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device), label_smoothing=0.1
    )
    optimizer = optim.NAdam(
        model.parameters(),
        lr=7e-4,
        weight_decay=1e-4
    )
    early_stopping = EarlyStopping(patience=5)
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses   = []

    # ── EPOCH LOOP ────────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):

        print(f"\nEpoch {epoch + 1}/{EPOCHS}")

        # TRAIN
        model.train()
        train_loss = 0

        for images, lbls in tqdm(train_loader, desc="Train"):

            images = images.to(device)
            lbls   = lbls.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss    = criterion(outputs, lbls)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc="Val"):
                images, targets = images.to(device), targets.to(device)
                outputs   = model(images)
                val_loss += criterion(outputs, targets).item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(targets.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)

        # METRICS
        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average='weighted', zero_division=0)
        recall    = recall_score(trues, preds, average='weighted', zero_division=0)
        f1        = f1_score(trues, preds, average='weighted', zero_division=0)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        # ── WANDB LOG PER EPOCH ───────────────────────────────────────────────
        # Panel Loss      → fold_N/train_loss, fold_N/val_loss
        # Panel Accuracy  → fold_N/accuracy
        # Panel Precision → fold_N/precision
        # Panel Recall    → fold_N/recall
        # Panel F1 Score  → fold_N/f1_score
        # Panel LR        → fold_N/lr
        # Semua pakai key "epoch" sebagai x-axis bersama
        wandb.log({
            "epoch"                    : epoch + 1,

            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss"  : avg_val_loss,

            f"fold_{fold+1}/accuracy"  : acc,
            f"fold_{fold+1}/precision" : precision,
            f"fold_{fold+1}/recall"    : recall,
            f"fold_{fold+1}/f1_score"  : f1,

            f"fold_{fold+1}/lr"        : optimizer.param_groups[0]['lr'],
        })

        # SAVE BEST MODEL
        if avg_val_loss < best_val_loss:
            best_val_loss   = avg_val_loss
            best_train_loss = avg_train_loss

            save_path = f"/kaggle/working/model_fold_{fold + 1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss"        : avg_val_loss,
                "f1"              : f1,
                "fold"            : fold + 1
            }, save_path)
            best_model_path = save_path
            best_model_wts  = copy.deepcopy(model.state_dict())
            print(f"  ✓ Model saved → {save_path}")

        if early_stopping.step(avg_val_loss):
            print("Early Stopping Triggered")
            break

    # ── SIMPAN HISTORY ────────────────────────────────────────────────────────
    all_train_losses[fold + 1] = train_losses
    all_val_losses[fold + 1]   = val_losses

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax    = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label='Train Loss', marker='o', markersize=3)
    ax.plot(epochs_ran, val_losses,   label='Val Loss',   marker='o', markersize=3)
    ax.set_title(f'Fold {fold + 1} — Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    curve_path = f"/kaggle/working/Fold_{fold + 1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches='tight')
    wandb.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)
    print(f"  ✓ Loss curve saved → {curve_path}")

    # ── UPLOAD MODEL ARTIFACT ─────────────────────────────────────────────────
    if best_model_path:
        artifact = wandb.Artifact(name=f"model-fold-{fold+1}", type="model")
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
        all_fold_best_paths.append(best_model_path)

    # ── FINAL EVALUATION FOLD (pakai best model) ──────────────────────────────
    if best_model_path:
        model.load_state_dict(
            torch.load(best_model_path, map_location=device)["model_state_dict"]
        )

    model.eval()
    final_preds, final_trues = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            final_preds.extend(model(images).argmax(1).cpu().numpy())
            final_trues.extend(targets.cpu().numpy())

    print("\nClassification Report")
    print(classification_report(final_trues, final_preds, target_names=classes, zero_division=0))

    fold_acc  = accuracy_score(final_trues, final_preds)
    fold_prec = precision_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_rec  = recall_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_f1_  = f1_score(final_trues, final_preds, average='weighted', zero_division=0)

    fold_accuracies.append(fold_acc)
    fold_precision.append(fold_prec)
    fold_recall.append(fold_rec)
    fold_f1.append(fold_f1_)

    fold_results.append({
        "Fold": fold + 1,
        "Train_Loss": best_train_loss,
        "Val_Loss": best_val_loss,
        "Accuracy": fold_acc,
        "Precision": fold_prec,
        "Recall": fold_rec,
        "F1": fold_f1_
    })

    # ── WANDB LOG FINAL METRICS FOLD ─────────────────────────────────────────
    wandb.log({
        f"fold_{fold+1}/final_accuracy" : fold_acc,
        f"fold_{fold+1}/final_precision": fold_prec,
        f"fold_{fold+1}/final_recall"   : fold_rec,
        f"fold_{fold+1}/final_f1"       : fold_f1_,

        f"fold_{fold+1}/confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=final_trues,
            preds=final_preds,
            class_names=classes
        )
    })

    print(f"\nFold {fold+1} selesai — Acc: {fold_acc:.4f} | F1: {fold_f1_:.4f}")


results_df = pd.DataFrame(fold_results)

results_df.loc[len(results_df)] = {
    "Fold": "Mean",
    "Train_Loss": results_df["Train_Loss"].mean(),
    "Val_Loss": results_df["Val_Loss"].mean(),
    "Accuracy": np.mean(fold_accuracies),
    "Precision": np.mean(fold_precision),
    "Recall": np.mean(fold_recall),
    "F1": np.mean(fold_f1)
}

csv_path = "/kaggle/working/KFold_Summary.csv"
results_df.to_csv(csv_path, index=False)

artifact = wandb.Artifact(
    "kfold-summary",
    type="results"
)

artifact.add_file(csv_path)

wandb.log_artifact(artifact)



  FOLD 1 / 5

Epoch 1/100


Val: 100%|██████████| 98/98 [00:42<00:00,  2.30it/s]


Train Loss : 3.2333 | Val Loss  : 3.3360
Accuracy   : 0.1144  | Precision : 0.0773
Recall     : 0.1144  | F1 Score  : 0.0637
  ✓ Model saved → /kaggle/working/model_fold_1.pth

Epoch 2/100


Val: 100%|██████████| 98/98 [00:24<00:00,  4.08it/s]


Train Loss : 3.1054 | Val Loss  : 3.1725
Accuracy   : 0.1665  | Precision : 0.1709
Recall     : 0.1665  | F1 Score  : 0.1280
  ✓ Model saved → /kaggle/working/model_fold_1.pth

Epoch 3/100


...

Val: 100%|██████████| 98/98 [00:25<00:00,  3.79it/s]


Train Loss : 2.1233 | Val Loss  : 2.6718
Accuracy   : 0.3787  | Precision : 0.4656
Recall     : 0.3787  | F1 Score  : 0.3929

Epoch 43/100


Val: 100%|██████████| 98/98 [00:26<00:00,  3.64it/s]


Train Loss : 2.1157 | Val Loss  : 2.6637
Accuracy   : 0.3922  | Precision : 0.4583
Recall     : 0.3922  | F1 Score  : 0.3982
Early Stopping Triggered
  ✓ Loss curve saved → /kaggle/working/Fold_5_Loss_Curve.png

Classification Report
                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.59      0.54      0.56       168
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.55      0.41      0.47       229
                                          Atopic Dermatitis Photos       0.40      0.40      0.40        98
                                            Bullous Disease Photos       0.30      0.39      0.34        90
                Cellulitis Impetigo and other Bacterial Infections       0.12      0.25      0.16        57
                                                     Eczema Photos       0.57      0.37      0.45       247
         

<Artifact kfold-summary>

# **Grafik Gabungan & Final Summary**

In [11]:
# ── GRAFIK GABUNGAN SEMUA FOLD ────────────────────────────────────────────────
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, fold_n in enumerate(all_train_losses.keys()):
    ep = range(1, len(all_train_losses[fold_n]) + 1)
    c  = colors[(fold_n - 1) % len(colors)]
    axes[0].plot(ep, all_train_losses[fold_n], label=f'Fold {fold_n}', color=c, marker='o', markersize=3)
    axes[1].plot(ep, all_val_losses[fold_n],   label=f'Fold {fold_n}', color=c, marker='o', markersize=3)

for ax, title in zip(axes, ['Train Loss — Semua Fold', 'Val Loss — Semua Fold']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Perbandingan Loss Semua Fold', fontsize=14, fontweight='bold')
plt.tight_layout()

combined_path = "/kaggle/working/All_Folds_Loss_Curve.png"
fig.savefig(combined_path, dpi=150, bbox_inches='tight')
wandb.log({"Loss_Curve/All_Folds_Combined": wandb.Image(combined_path)})
plt.close(fig)
print(f"✓ Grafik gabungan disimpan → {combined_path}")

# ── SUMMARY METRICS ───────────────────────────────────────────────────────────
print("\n" + "="*50)
print("  FINAL RESULT — ALL FOLDS")
print("="*50)
print(f"Mean Accuracy  : {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Mean Precision : {np.mean(fold_precision):.4f} ± {np.std(fold_precision):.4f}")
print(f"Mean Recall    : {np.mean(fold_recall):.4f} ± {np.std(fold_recall):.4f}")
print(f"Mean F1 Score  : {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")

# ── WANDB LOG SUMMARY ─────────────────────────────────────────────────────────
# Panel Summary → summary/mean_accuracy, summary/mean_precision, dst.
wandb.log({
    "summary/mean_accuracy"  : np.mean(fold_accuracies),
    "summary/mean_precision" : np.mean(fold_precision),
    "summary/mean_recall"    : np.mean(fold_recall),
    "summary/mean_f1"        : np.mean(fold_f1),
    "summary/std_accuracy"   : np.std(fold_accuracies),
    "summary/std_f1"         : np.std(fold_f1),
})



✓ Grafik gabungan disimpan → /kaggle/working/All_Folds_Loss_Curve.png

  FINAL RESULT — ALL FOLDS
Mean Accuracy  : 0.3676 ± 0.0179
Mean Precision : 0.4404 ± 0.0164
Mean Recall    : 0.3676 ± 0.0179
Mean F1 Score  : 0.3715 ± 0.0178


# **Test Evaluation**

In [12]:
best_overall_path = all_fold_best_paths[fold_f1.index(max(fold_f1))]
print(f"Best model path : {best_overall_path}")
print(f"Best F1         : {max(fold_f1):.4f}")

test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Best model path : /kaggle/working/model_fold_4.pth
Best F1         : 0.3893


DermNetCNN(
  (block1): ConvBlock(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): PReLU(num_parameters=1)
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): ConvBlock(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): PReLU(num_parameters=1)
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): ConvBlock(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): PReLU(num_parameters=1)
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): ConvBlock(
    (conv): C

# **Test Confusion Matrix**

In [13]:
y_true, y_pred = [], []

with torch.no_grad():
    for images, lbs in tqdm(test_loader, desc="Test"):
        images  = images.to(device)
        outputs = model(images)
        y_true.extend(lbs.cpu().numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())

acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm  = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(
    cmap='Blues', ax=ax, xticks_rotation=90
)
plt.tight_layout()
plt.savefig("/kaggle/working/Test_Confusion_Matrix.png", dpi=150, bbox_inches='tight')
plt.show()

wandb.log({
    "Test/Confusion_Matrix": wandb.Image(
        "/kaggle/working/Test_Confusion_Matrix.png"
    ),
    "test/accuracy": acc,
    "test/precision": precision,
    "test/recall": recall,
    "test/f1": f1
})

# ==========================================
# TEST RESULT CSV
# ==========================================

test_results_df = pd.DataFrame({
    "Filename": [test_dataset.samples[i][0] for i in range(len(y_true))],
    "True_Label": [classes[i] for i in y_true],
    "Predicted_Label": [classes[i] for i in y_pred],
    "Correct": np.array(y_true) == np.array(y_pred)
})


test_summary_df = pd.DataFrame([{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}])

summary_path = "/kaggle/working/Test_Summary.csv"
test_summary_df.to_csv(summary_path, index=False)

csv_test_path = "/kaggle/working/Test_Result.csv"
test_results_df.to_csv(csv_test_path, index=False)

print(f"Test CSV saved -> {csv_test_path}")


# ==========================================
# UPLOAD TEST CSV KE WANDB
# ==========================================

artifact = wandb.Artifact(
    name="test-results",
    type="results"
)

artifact.add_file(csv_test_path)
artifact.add_file(summary_path)

wandb.log_artifact(artifact)

wandb.finish()
print("\nWandB run selesai.")

Test: 100%|██████████| 126/126 [00:58<00:00,  2.17it/s]


Accuracy  : 0.3748
Precision : 0.4245
Recall    : 0.3748
F1-Score  : 0.3797

                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.52      0.64      0.57       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.59      0.36      0.45       288
                                          Atopic Dermatitis Photos       0.19      0.45      0.26       123
                                            Bullous Disease Photos       0.23      0.31      0.26       113
                Cellulitis Impetigo and other Bacterial Infections       0.16      0.22      0.19        73
                                                     Eczema Photos       0.53      0.37      0.43       309
                                      Exanthems and Drug Eruptions       0.31      0.26      0.28       101
                 Hair Loss Photos Alopecia and other Hair 

wandb: uploading artifact test-results; updating run metadata
wandb: uploading artifact test-results
wandb: uploading config.yaml; uploading media/images/Test/Confusion_Matrix_175_2c79b19fe44ff8af3ad7.png; uploading output.log; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:                  epoch ▁▂▃▄▄▁▂▂▃▄▆▆▂▂▃▃▄▄▄▅▂▃▃▃▄▄▄▅▅▅▃▃▃▄▅▅▆▇▇█
wandb:        fold_1/accuracy ▁▂▃▄▄▄▄▅▅▅▄▆▅▆▆▅▆▆▇▇▇▆▆▇██▇▇▇██
wandb:        fold_1/f1_score ▁▂▃▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▆▇███▇▇██
wandb:  fold_1/final_accuracy ▁
wandb:        fold_1/final_f1 ▁
wandb: fold_1/final_precision ▁
wandb:    fold_1/final_recall ▁
wandb:              fold_1/lr ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       fold_1/precision ▁▃▅▄▅▅▅▅▅▆▇▆▆▆▇▆▆▇▇▇▇▇▇█▇██████
wandb:          fold_1/recall ▁▂▃▄▄▄▄▅▅▅▄▆▅▆▆▅▆▆▇▇▇▆▆▇██▇▇▇██
wandb:                    +56 ...
wandb: 
wandb: Run summary:
wandb:                  epoch 43
wandb:        fold_1/accuracy 0.35411
wandb:        fold_1/f1_score 0.36697
wandb:  fold_1/final_accuracy 0.371


WandB run selesai.
